# MNIST CNN accelerator for the DE10-Lite — Python phase

Trains a small float CNN on MNIST, quantises it to int8 (per-channel weights, fixed-point multiply-then-shift requant), and exports everything the RTL phase needs: weight `.mif` files, per-layer **golden vectors**, the 30-image demo `.mif`, and the matching PNGs.

**The int8 NumPy model here is the exact specification the hardware must reproduce bit-for-bit.** Run the cells top to bottom. GPU runtime recommended but not required (trains in a couple of minutes on CPU).

_Tip: Runtime → Change runtime type → GPU._

## 0. Setup

In [ ]:
# ---- Colab setup ----------------------------------------------------------
# TensorFlow and NumPy are preinstalled on Colab. Pillow usually is too.
import sys, subprocess, os, json, numpy as np
try:
    import PIL
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow"])
for d in ("outputs/weights", "outputs/golden", "outputs/images_png"):
    os.makedirs(d, exist_ok=True)
SEED = 0
print("NumPy", np.__version__)


## 1. The integer golden model (the RTL specification)

Pure NumPy. Exact rounding, int32 accumulator, saturation, per-channel requant — this is what the SystemVerilog must match element-for-element.

In [ ]:
"""
step2_quantize.py  --  Per-channel PTQ and the bit-exact integer "golden" model.

This module is the SPECIFICATION the RTL must reproduce exactly.  Every integer
op here (rounding, shift, saturation, accumulator width) is what the hardware
implements.  There is deliberately NO TensorFlow here -- pure NumPy integers.

Quantisation scheme (uniform affine, symmetric, zero_point = 0):
  * weights     : int8, symmetric, PER-CHANNEL (one scale per output channel)
                  s_w[c] = max|W[...,c]| / 127
  * activations : int8, symmetric, PER-TENSOR, post-ReLU so values land in [0,127]
                  s_a = max|a| / 127  (calibrated on a small train subset)
  * input image : raw uint8 [0,255], s_in = 1/255 (the 1/255 normalisation is
                  FOLDED into layer-1's requant multiplier -- no divide on-chip)
  * bias        : int32, quantised at s_in*s_w[c] so it adds in accumulator units
  * accumulator : int32 (a full dot product never overflows before requant)
  * requant     : fixed-point multiply-then-shift with rounding, per the brief:
                  out = sat( (acc * M0[c] + (1<<(n-1))) >> n )
                  where M[c] = s_in_layer * s_w[c] / s_out_layer ~= M0[c] / 2^n
"""
import numpy as np

INT8_MIN, INT8_MAX = -128, 127


# ---- quantiser helpers -----------------------------------------------------
def per_channel_wscale(W_co_last):
    """W shape (..., out_ch).  Returns (q_int8, scale_per_ch)."""
    axes = tuple(range(W_co_last.ndim - 1))
    s = np.maximum(np.abs(W_co_last).max(axis=axes), 1e-12) / 127.0
    q = np.clip(np.rint(W_co_last / s), INT8_MIN, INT8_MAX).astype(np.int32)
    return q, s.astype(np.float64)


def choose_M0_shift(M):
    """Approximate real per-channel multipliers M[c] (each < ~a few) by M0[c]/2^n
    with a single per-layer shift n.  n is picked so the largest M0 uses ~15 bits
    (int16-friendly, plenty of precision)."""
    n = int(np.floor(np.log2(32767.0 / max(M.max(), 1e-12))))
    n = int(np.clip(n, 0, 31))
    M0 = np.rint(M * (2.0 ** n)).astype(np.int64)
    return M0, n


def requant(acc, M0, n, relu):
    """Integer requant: (acc*M0 + round) >> n, then saturate to int8.
    acc, M0 broadcast per output channel.  Arithmetic (floor) shift."""
    rnd = (1 << (n - 1)) if n > 0 else 0
    out = (acc * M0 + rnd) >> n            # numpy >> on int64 is arithmetic floor
    lo = 0 if relu else INT8_MIN           # post-ReLU is non-negative
    return np.clip(out, lo, INT8_MAX).astype(np.int32)


# ---- primitive layers (integer) --------------------------------------------
def conv2d_int(x, w_q, stride=1):
    """x:(H,W,Cin) int, w_q:(kh,kw,Cin,Cout) int8.  Valid cross-correlation.
    Returns int32 accumulator (H-kh+1, W-kw+1, Cout).
    Vectorised with as_strided -- numerically IDENTICAL to the naive nested-loop
    MAC the RTL performs (integer einsum), just faster for batch evaluation."""
    from numpy.lib.stride_tricks import as_strided
    x = np.ascontiguousarray(x.astype(np.int64))
    H, Wd, Ci = x.shape
    kh, kw, _, Co = w_q.shape
    Ho, Wo = H - kh + 1, Wd - kw + 1
    s0, s1, s2 = x.strides
    patches = as_strided(x, (Ho, Wo, kh, kw, Ci), (s0, s1, s0, s1, s2))
    return np.einsum('ijhwc,hwco->ijo', patches, w_q.astype(np.int64))


def maxpool_int(x, k=2, s=2):
    H, Wd, C = x.shape
    Ho, Wo = H // s, Wd // s
    out = np.zeros((Ho, Wo, C), x.dtype)
    for i in range(Ho):
        for j in range(Wo):
            out[i, j] = x[i*s:i*s+k, j*s:j*s+k, :].reshape(-1, C).max(0)
    return out


def dense_int(x_vec, w_q):
    """x_vec:(N,) int, w_q:(N,Cout) int8 -> int32 acc (Cout,)."""
    return x_vec.astype(np.int64) @ w_q.astype(np.int64)


# ---- calibration: pick per-tensor activation scales from float activations --
def calibrate(fw, x_uint8_calib):
    """Run the FLOAT network on a calibration set, record max|activation| after
    each ReLU / at the logits, to set per-tensor output scales."""
    xs = x_uint8_calib.astype(np.float64) / 255.0
    mx = {k: 0.0 for k in ["conv1", "conv2", "dense1", "dense2"]}
    for img in xs:
        a = img[..., None]                                    # (28,28,1)
        a = _fconv(a, fw["conv1_w"], fw["conv1_b"]);  a = np.maximum(a, 0); mx["conv1"]=max(mx["conv1"], a.max())
        a = _fpool(a)
        a = _fconv(a, fw["conv2_w"], fw["conv2_b"]);  a = np.maximum(a, 0); mx["conv2"]=max(mx["conv2"], a.max())
        a = _fpool(a)
        v = a.reshape(-1)
        d = v @ fw["dense1_w"] + fw["dense1_b"]; d = np.maximum(d, 0); mx["dense1"]=max(mx["dense1"], np.abs(d).max())
        o = d @ fw["dense2_w"] + fw["dense2_b"];  mx["dense2"]=max(mx["dense2"], np.abs(o).max())
    return {k: v / 127.0 for k, v in mx.items()}              # s_out per layer


# float helpers used only for calibration + the float<->int sanity check
def _fconv(x, w, b):
    from numpy.lib.stride_tricks import as_strided
    x = np.ascontiguousarray(x, dtype=np.float64)
    H, Wd, Ci = x.shape; kh, kw, _, Co = w.shape
    Ho, Wo = H-kh+1, Wd-kw+1
    s0, s1, s2 = x.strides
    patches = as_strided(x, (Ho, Wo, kh, kw, Ci), (s0, s1, s0, s1, s2))
    return np.einsum('ijhwc,hwco->ijo', patches, w) + b
def _fpool(x):
    H, Wd, C = x.shape; out = np.zeros((H//2, Wd//2, C))
    for i in range(H//2):
        for j in range(Wd//2):
            out[i, j] = x[2*i:2*i+2, 2*j:2*j+2, :].reshape(-1, C).max(0)
    return out


# ---- build the quantised parameter set -------------------------------------
def quantize_params(fw, s_out):
    """Returns a dict of int weights/biases + per-channel (M0, shift) per layer."""
    P, s_in = {}, 1.0 / 255.0
    scale_in = {"conv1": s_in, "conv2": s_out["conv1"],
                "dense1": s_out["conv2"], "dense2": s_out["dense1"]}
    for name, wk, bk in [("conv1", "conv1_w", "conv1_b"),
                         ("conv2", "conv2_w", "conv2_b"),
                         ("dense1", "dense1_w", "dense1_b"),
                         ("dense2", "dense2_w", "dense2_b")]:
        wq, sw = per_channel_wscale(fw[wk])
        sin = scale_in[name]
        bq = np.rint(fw[bk] / (sin * sw)).astype(np.int64)     # int32 bias
        M = sin * sw / s_out[name]                             # real multiplier/ch
        M0, n = choose_M0_shift(M)
        P[name] = dict(wq=wq, bq=bq, M0=M0, shift=n, s_w=sw,
                       s_in=sin, s_out=s_out[name])
    P["_scales"] = dict(s_in=s_in, **{f"s_out_{k}": v for k, v in s_out.items()})
    return P


# ---- the GOLDEN integer forward pass (this === the RTL) ---------------------
def infer_int(P, img_uint8, dump=None):
    """Full integer inference on one raw-uint8 28x28 image. Returns predicted
    label.  If `dump` is a dict, stores every layer's integer output for the
    golden vectors / scoreboard."""
    a = img_uint8.astype(np.int64)[..., None]                 # (28,28,1) uint8 acts

    acc = conv2d_int(a, P["conv1"]["wq"]) + P["conv1"]["bq"]
    c1  = requant(acc, P["conv1"]["M0"], P["conv1"]["shift"], relu=True)
    p1  = maxpool_int(c1)

    acc = conv2d_int(p1, P["conv2"]["wq"]) + P["conv2"]["bq"]
    c2  = requant(acc, P["conv2"]["M0"], P["conv2"]["shift"], relu=True)
    p2  = maxpool_int(c2)

    flat = p2.reshape(-1)                                      # (h*W+w)*C+c order
    acc = dense_int(flat, P["dense1"]["wq"]) + P["dense1"]["bq"]
    d1  = requant(acc, P["dense1"]["M0"], P["dense1"]["shift"], relu=True)

    acc = dense_int(d1, P["dense2"]["wq"]) + P["dense2"]["bq"]
    logits = requant(acc, P["dense2"]["M0"], P["dense2"]["shift"], relu=False)
    pred = int(np.argmax(logits))

    if dump is not None:
        dump.update(conv1=c1, pool1=p1, conv2=c2, pool2=p2,
                    flatten=flat, dense1=d1, logits=logits, pred=pred)
    return pred


## 2. Train the float CNN (Keras)

The only TensorFlow step. Produces the float weights; saves a held-out **train** subset for activation calibration so the test set stays untouched.

In [ ]:
import os, numpy as np

SEED = 0   # set for reproducible weights across runs (so a re-export is stable)


# ----------------------------------------------------------------------------
# Data.  Standard MNIST as raw uint8 images [0,255], shape (N,28,28).
# Primary path is the Keras built-in (works on Colab). If the dataset download
# is unavailable (e.g. an offline sandbox), fall back to a local mnist.pkl.gz.
# ----------------------------------------------------------------------------
def load_mnist():
    try:
        from tensorflow import keras
        (xtr, ytr), (xte, yte) = keras.datasets.mnist.load_data()   # uint8 [0,255]
        return ((xtr.astype(np.uint8), ytr.astype(np.int64)),
                (xte.astype(np.uint8), yte.astype(np.int64)))
    except Exception as e:
        print(f"[data] Keras MNIST unavailable ({e}); falling back to mnist.pkl.gz")
        import gzip, pickle
        tr, va, te = pickle.load(gzip.open("mnist.pkl.gz", "rb"), encoding="latin1")
        def u8(s):
            return np.rint(s[0] * 256.0).astype(np.uint8).reshape(-1, 28, 28), s[1].astype(np.int64)
        (xtr, ytr), (xva, yva), (xte, yte) = map(u8, (tr, va, te))
        xtr = np.concatenate([xtr, xva]); ytr = np.concatenate([ytr, yva])   # -> 60k
        return (xtr, ytr), (xte, yte)


def build_model():
    from tensorflow import keras
    from tensorflow.keras import layers
    # 'valid' padding => the 28->26->13->11->5 shapes in the brief.
    m = keras.Sequential([
        keras.Input((28, 28, 1)),
        layers.Conv2D(8,  3, padding="valid", activation="relu", name="conv1"),
        layers.MaxPooling2D(2, name="pool1"),
        layers.Conv2D(16, 3, padding="valid", activation="relu", name="conv2"),
        layers.MaxPooling2D(2, name="pool2"),
        layers.Flatten(name="flatten"),                      # channels_last: (h*W+w)*C+c
        layers.Dense(64, activation="relu", name="dense1"),
        layers.Dense(10, activation=None,  name="dense2"),   # logits
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
    return m


def train_main():
    from tensorflow import keras
    keras.utils.set_random_seed(SEED)                        # reproducible weights

    (xtr, ytr), (xte, yte) = load_mnist()
    m = build_model()
    xtr_n = (xtr.astype(np.float32) / 255.0)[..., None]      # s_in = 1/255
    xte_n = (xte.astype(np.float32) / 255.0)[..., None]
    m.fit(xtr_n, ytr, validation_data=(xte_n, yte),
          epochs=10, batch_size=128, verbose=2)
    _, acc = m.evaluate(xte_n, yte, verbose=0)
    print(f"[float] test accuracy = {acc*100:.2f}%")

    # Persist float weights (plain NumPy), the uint8 test set, and a small TRAIN
    # subset used only to calibrate activation scales (never evaluated on).
    os.makedirs("outputs", exist_ok=True)
    w = {L.name: [a.astype(np.float32) for a in L.get_weights()]
         for L in m.layers if L.get_weights()}
    np.savez("outputs/float_weights.npz",
             conv1_w=w["conv1"][0], conv1_b=w["conv1"][1],
             conv2_w=w["conv2"][0], conv2_b=w["conv2"][1],
             dense1_w=w["dense1"][0], dense1_b=w["dense1"][1],
             dense2_w=w["dense2"][0], dense2_b=w["dense2"][1],
             float_acc=np.float32(acc))
    np.savez("outputs/mnist_test.npz", x=xte, y=yte)         # uint8 test images + labels
    np.savez("outputs/calib.npz", x=xtr[:1000])              # train subset for calibration
    print("[float] saved float_weights.npz, mnist_test.npz, calib.npz")

train_main()


## 3. Quantise, evaluate on the full 10k test set, and export artifacts

Reports float vs int8 accuracy, checks the NumPy float path against Keras, then writes weights/golden/images/PNGs. Expect the int8 accuracy within ~0.1 pt of float, int-vs-float agreement ~100%, and the 30 demo predictions running 0,0,0,…,9,9,9.

In [ ]:
import os, json, numpy as np

OUT = "outputs"


# ---------- .mif writers -----------------------------------------------------
def _twos_hex(v, bits):
    return format(int(v) & ((1 << bits) - 1), f"0{bits // 4}x")

def write_mif(path, values, width):
    values = list(values)
    with open(path, "w") as f:
        f.write(f"WIDTH={width};\nDEPTH={len(values)};\n")
        f.write("ADDRESS_RADIX=DEC;\nDATA_RADIX=HEX;\nCONTENT BEGIN\n")
        for a, v in enumerate(values):
            f.write(f"    {a} : {_twos_hex(v, width)};\n")
        f.write("END;\n")


# ---------- weight export (documented memory order) --------------------------
def export_weights(P):
    d = f"{OUT}/weights"
    # Conv weights wq:(kh,kw,Cin,Cout).  Order emitted: co -> ci -> ky -> kx.
    for name in ("conv1", "conv2"):
        wq = P[name]["wq"]; kh, kw, Ci, Co = wq.shape
        seq = [wq[ky, kx, ci, co]
               for co in range(Co) for ci in range(Ci)
               for ky in range(kh) for kx in range(kw)]
        write_mif(f"{d}/{name}_w.mif", seq, 8)
        write_mif(f"{d}/{name}_b.mif", P[name]["bq"], 32)   # int32 bias, co order
    # Dense weights wq:(Nin,Cout).  Order emitted: co -> ni  (each output's vector contiguous)
    for name in ("dense1", "dense2"):
        wq = P[name]["wq"]; Ni, Co = wq.shape
        seq = [wq[ni, co] for co in range(Co) for ni in range(Ni)]
        write_mif(f"{d}/{name}_w.mif", seq, 8)
        write_mif(f"{d}/{name}_b.mif", P[name]["bq"], 32)


def export_params_json(P):
    def L(name):
        p = P[name]
        return dict(shift=int(p["shift"]),
                    M0_per_channel=[int(x) for x in p["M0"]],
                    weight_scale_per_channel=[float(x) for x in p["s_w"]],
                    s_in=float(p["s_in"]), s_out=float(p["s_out"]),
                    bias_int32=[int(x) for x in p["bq"]])
    doc = {
        "_comment": "Golden quant params. requant: out=sat((acc*M0[c]+(1<<(n-1)))>>n); "
                    "acc int32; weights int8 sym per-channel; acts int8 sym per-tensor "
                    "post-ReLU [0,127]; input uint8 [0,255] with s_in=1/255 folded into conv1.",
        "flatten_order": "(h*W + w)*C + c   (channels_last, matches Keras Flatten)",
        "layers": {n: L(n) for n in ("conv1", "conv2", "dense1", "dense2")},
    }
    json.dump(doc, open(f"{OUT}/quant_params.json", "w"), indent=2)


# ---------- demo image selection + export -----------------------------------
def select_demo_images(P, x_uint8, y):
    """3 correctly-classified, clear examples per digit -> 30 images.
    Layout / switch index:  index = digit*3 + variation  (0..29)."""
    chosen = []
    for digit in range(10):
        idxs = np.where(y == digit)[0]
        picks = []
        for i in idxs:
            if infer_int(P, x_uint8[i]) == digit:     # golden predicts correctly
                picks.append(i)
            if len(picks) == 3:
                break
        chosen += picks
    return np.array(chosen)                              # 30 indices


def export_images(x_uint8, demo_idx):
    from PIL import Image
    imgs = x_uint8[demo_idx]                             # (30,28,28) uint8
    # images.mif : addr = img*784 + (row*28 + col)
    seq = [int(px) for img in imgs for px in img.reshape(-1)]
    write_mif(f"{OUT}/images.mif", seq, 8)
    for k, img in enumerate(imgs):
        digit, var = k // 3, k % 3
        Image.fromarray(img).save(   # 2-D uint8 -> mode 'L' inferred (no deprecated arg)
            f"{OUT}/images_png/idx{k:02d}_digit{digit}_v{var}.png")


def export_golden(P, x_uint8, demo_idx):
    """Per-layer integer golden vectors for the 30 demo images (scoreboard input)."""
    d = f"{OUT}/golden"
    preds = []
    for k, gi in enumerate(demo_idx):
        dump = {}
        infer_int(P, x_uint8[gi], dump=dump)
        preds.append(dump["pred"])
        with open(f"{d}/img{k:02d}_layers.txt", "w") as f:
            f.write(f"# demo index {k}  (digit {k//3}, variation {k%3})  "
                    f"pred={dump['pred']}\n")
            for lname in ("conv1", "pool1", "conv2", "pool2",
                          "flatten", "dense1", "logits"):
                arr = np.asarray(dump[lname]).reshape(-1)
                f.write(f"# {lname}  shape={np.asarray(dump[lname]).shape}  n={arr.size}\n")
                for v in arr:
                    f.write(f"{int(v)}\n")
    np.savetxt(f"{d}/predicted_labels.txt", np.array(preds, int), fmt="%d")
    return preds


# ---------- accuracy on full 10k test set -----------------------------------
def evaluate_int(P, x_uint8, y, n=None):
    n = len(x_uint8) if n is None else n
    correct = sum(infer_int(P, x_uint8[i]) == y[i] for i in range(n))
    return correct / n


# ---------- main -------------------------------------------------------------
def export_main():
    fw = dict(np.load(f"{OUT}/float_weights.npz"))
    float_acc = float(fw.pop("float_acc"))
    test = np.load(f"{OUT}/mnist_test.npz"); xte, yte = test["x"], test["y"]

    # 1) calibrate activation scales on a small TRAIN subset (saved by step1);
    #    the test set stays fully held out.  1000 images is plenty for MNIST.
    calib = np.load(f"{OUT}/calib.npz")["x"]
    s_out = calibrate(fw, calib)

    # 2) build quantised params
    P = quantize_params(fw, s_out)

    # 3) sanity: NumPy FLOAT forward must match Keras predictions (validates the
    #    golden model's structure/flatten order before we trust the int path)
    #    -- compare argmax on 200 images
    def float_pred(img):
        a = img.astype(np.float64)[..., None] / 255.0
        a = _fconv(a, fw["conv1_w"], fw["conv1_b"]); a = np.maximum(a, 0); a = _fpool(a)
        a = _fconv(a, fw["conv2_w"], fw["conv2_b"]); a = np.maximum(a, 0); a = _fpool(a)
        v = a.reshape(-1)
        v = np.maximum(v @ fw["dense1_w"] + fw["dense1_b"], 0)
        return int(np.argmax(v @ fw["dense2_w"] + fw["dense2_b"]))
    fp = np.array([float_pred(xte[i]) for i in range(200)])

    # 4) accuracy of the INT golden model on the full 10k test set
    int_acc = evaluate_int(P, xte, yte)

    # 5) export everything
    export_weights(P)
    export_params_json(P)
    demo_idx = select_demo_images(P, xte, yte)
    export_images(xte, demo_idx)
    preds = export_golden(P, xte, demo_idx)

    # agreement of int golden vs float on the 200-sample check
    ip = np.array([infer_int(P, xte[i]) for i in range(200)])
    agree = (fp == ip).mean()

    with open(f"{OUT}/REPORT.md", "w") as f:
        f.write("# MNIST FPGA accelerator - Python phase report\n\n")
        f.write(f"- Float model test accuracy (10,000 images): **{float_acc*100:.2f}%**\n")
        f.write(f"- Int8 golden model test accuracy (10,000 images): **{int_acc*100:.2f}%**\n")
        f.write(f"- Float->int8 drop: **{(float_acc-int_acc)*100:.2f} pts**\n")
        f.write(f"- Int vs float prediction agreement (200-sample sanity): {agree*100:.1f}%\n\n")
        f.write("Quantisation: int8 symmetric per-channel weights, int8 per-tensor "
                "activations, int32 accumulator, fixed-point multiply-then-shift requant "
                "with rounding, 1/255 folded into conv1. This int8 result is the number "
                "the hardware reproduces bit-for-bit; on-hardware accuracy = int8 accuracy "
                "by equivalence.\n\n")
        f.write("## Demo\n30 images (3 per digit), switch index = digit*3 + variation.\n")
    print(f"[float] {float_acc*100:.2f}%   [int8] {int_acc*100:.2f}%   "
          f"[drop] {(float_acc-int_acc)*100:.2f}pts   [int/float agree] {agree*100:.1f}%")
    print("demo indices:", demo_idx.tolist())
    print("demo predictions:", preds)

export_main()


## 4. Download the artifacts

In [ ]:
# ---- Zip the artifacts and download ---------------------------------------
import shutil
shutil.make_archive("mnist_fpga_python_phase", "zip", "outputs")
print("Zipped outputs/ -> mnist_fpga_python_phase.zip")
try:
    from google.colab import files
    files.download("mnist_fpga_python_phase.zip")
except Exception as e:
    print("(not on Colab, skipping auto-download)", e)
